# 02 — Build Feature Cache (run once)

This notebook preprocesses every audio+MIDI pair in MAESTRO and saves the
results as compressed `.npz` files on Google Drive.

**Why cache?**  
Loading raw audio → mel spectrogram → piano rolls on-the-fly during training
would be a bottleneck.  Precomputing and caching (NPZ strategy from
jongwook/onsets-and-frames) lets the DataLoader simply load pre-computed
tensors and crop them randomly.

**Run this once.** The cache stays on Drive and survives Colab restarts.  
**Expected time**: ~30 minutes for the full MAESTRO v3 dataset on a T4 GPU.  
**Expected size**: ~12–15 GB on Drive (compressed NPZ files).

## Setup

In [ ]:
import sys, os

REPO_DIR     = '/content/piano_amt'
DRIVE_ROOT   = '/content/drive/MyDrive/piano_amt'
MAESTRO_ROOT = f'{DRIVE_ROOT}/maestro-v3.0.0'
CACHE_DIR    = f'{DRIVE_ROOT}/cache'

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.makedirs(CACHE_DIR, exist_ok=True)
print(f"MAESTRO root: {MAESTRO_ROOT}")
print(f"Cache dir   : {CACHE_DIR}")

## Estimate cache size

In [ ]:
import glob
import pandas as pd

csv_files = sorted(glob.glob(f"{MAESTRO_ROOT}/*.csv"))
df = pd.read_csv(csv_files[0])

n_train = len(df[df["split"] == "train"])
n_val   = len(df[df["split"] == "validation"])
n_test  = len(df[df["split"] == "test"])
n_total = n_train + n_val + n_test

total_hrs = df["duration"].sum() / 3600

print(f"Files: train={n_train}, val={n_val}, test={n_test}, total={n_total}")
print(f"Total audio duration: {total_hrs:.1f} hours")
print(f"Estimated cache size: ~{total_hrs * 120:.0f} MB  (≈120 MB/hr for NPZ)")
print()
print("Note: Actual size depends on piece length. 15 GB is a safe upper bound.")

## Quick test on 3 files first

Always verify on a small subset before running the full cache build.

In [ ]:
import numpy as np
from pathlib import Path
from src.dataset import preprocess_and_cache, load_from_cache, _cache_path
from src.constants import (
    MAESTRO_AUDIO_COL, MAESTRO_MIDI_COL, MAESTRO_SPLIT_COL, N_MELS
)

train_rows = df[df[MAESTRO_SPLIT_COL] == "train"].head(3)

for _, row in train_rows.iterrows():
    audio_path = str(Path(MAESTRO_ROOT) / row[MAESTRO_AUDIO_COL])
    midi_path  = str(Path(MAESTRO_ROOT) / row[MAESTRO_MIDI_COL])
    cp         = _cache_path(audio_path, CACHE_DIR)

    print(f"\nProcessing: {Path(audio_path).name}")
    if not cp.exists():
        preprocess_and_cache(audio_path, midi_path, cp)
        print(f"  Cached → {cp}")
    else:
        print(f"  Already cached: {cp}")

    # Verify the cache loads correctly
    data = load_from_cache(cp)
    mel = data['mel']  # (229, T)
    assert mel.shape[0] == N_MELS, f"mel.shape[0]={mel.shape[0]}, expected {N_MELS}"
    assert data['onset'].shape[1] == 88
    print(f"  mel.shape    = {tuple(mel.shape)}")
    print(f"  onset.shape  = {tuple(data['onset'].shape)}")
    print(f"  mel range    = [{mel.min():.2f}, {mel.max():.2f}]")

print("\n✓ Quick test passed — safe to build full cache.")

## Build full cache

This cell processes all train / validation / test files.  
**Expected time: ~25–40 minutes on T4 GPU.**  
Already-cached files are automatically skipped, so it is safe to interrupt
and resume.

In [ ]:
from src.dataset import build_cache

build_cache(
    maestro_root=MAESTRO_ROOT,
    cache_dir=CACHE_DIR,
    splits=("train", "validation", "test"),
)

print("\n✓ Full cache build complete!")

## Verify cache

In [ ]:
import glob
import numpy as np

npz_files = sorted(glob.glob(f"{CACHE_DIR}/*.npz"))
print(f"Total .npz files in cache: {len(npz_files)}")

# Load and inspect the first file
if npz_files:
    sample_file = npz_files[0]
    print(f"\nInspecting: {sample_file}")
    data = np.load(sample_file)
    for k in data.files:
        v = data[k]
        print(f"  {k:12s}: shape={v.shape}  dtype={v.dtype}")

    mel = data["mel"]
    assert mel.shape[0] == 229, f"mel.shape[0]={mel.shape[0]}, expected 229"
    assert data["onset"].shape[1] == 88
    print(f"\n✓ Cache structure verified.")
else:
    print("WARNING: No .npz files found in cache!")

## Cache is complete — you never need to run this notebook again

The `.npz` files are stored on Google Drive at:
```
/content/drive/MyDrive/piano_amt/cache/
```

Every future Colab session starts from `03_verify_pipeline.ipynb` — which
mounts Drive and loads directly from the cache without re-running this notebook.

If you ever need to regenerate the cache (e.g. after changing mel parameters),
delete the `cache/` folder on Drive and re-run this notebook.